# Symbolic Music Dataset Exploration

This notebook introduces a small symbolic melody dataset for teaching sequence models.

Each melody is a sequence of note-duration tokens such as:
- `C4_q` for middle C as a quarter note
- `F#4_h` for F sharp as a half note
- `REST_e` for an eighth-note rest


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "nirma_university_language_models").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [ ]:
import matplotlib.pyplot as plt

from nirma_university_language_models import (
    build_music_vocabulary,
    decode_music_ids,
    encode_music_tokens,
    flatten_music_token_sequences,
    load_music_token_sequences,
    melody_lengths,
    melody_tokens_to_waveform,
    most_common_music_tokens,
)


In [ ]:
melodies, data_path = load_music_token_sequences()
print("Dataset path:", data_path)
print("Number of melodies:", len(melodies))
print("First melody:", melodies[0])

## 1. Melody lengths

Unlike raw audio, symbolic music is compact and easy to inspect.
Each line in the dataset is one melody, and each token is one musical event.

In [ ]:
lengths = melody_lengths(melodies)
print("Minimum length:", min(lengths))
print("Maximum length:", max(lengths))
print("Average length:", sum(lengths) / len(lengths))

plt.figure(figsize=(8, 4))
plt.hist(lengths, bins=8, edgecolor="black")
plt.title("Distribution of Melody Lengths")
plt.xlabel("Tokens per melody")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 2. Vocabulary

The next code cell builds an integer vocabulary exactly like a character-level language model.
The only difference is that our atomic units are music tokens instead of text characters.

In [ ]:
vocab, token_to_idx, idx_to_token = build_music_vocabulary(melodies)
print("Vocabulary size:", len(vocab))
print("Vocabulary sample:", vocab[:15])
print("Most common tokens:", most_common_music_tokens(melodies, limit=10))

In [ ]:
flat_tokens = flatten_music_token_sequences(melodies)
encoded = encode_music_tokens(flat_tokens, token_to_idx)
decoded = decode_music_ids(encoded[:20], idx_to_token)

print("Encoded sample:", encoded[:20])
print("Decoded sample:", decoded)

## 3. Where does embedding dimension come from?

The music model will also begin with an embedding layer, and the same rule applies here.

- `vocab_size` comes from the symbolic music dataset and counts unique note-duration tokens
- `embedding_dim` is chosen by us as a hyperparameter for the model
- the embedding layer shape is `(vocab_size, embedding_dim)`
- each token such as `C4_q` or `REST_e` is mapped to one dense vector of length `embedding_dim`

So the dataset decides how many different token rows we need, while the model designer decides how wide each learned vector should be.

In [ ]:
import torch

EMBED_DIM = 24
embedding = torch.nn.Embedding(num_embeddings=len(vocab), embedding_dim=EMBED_DIM)
sample_music_ids = torch.tensor(encoded[:8], dtype=torch.long)
sample_embeddings = embedding(sample_music_ids)

print("Embedding table shape:", tuple(embedding.weight.shape))
print("Input ID shape:", tuple(sample_music_ids.shape))
print("Output embedding shape:", tuple(sample_embeddings.shape))
print("First music-token embedding vector:", sample_embeddings[0])

## 4. Waveform preview

This repo keeps the model symbolic, but we can still synthesize a simple sine-wave preview.
That is enough for students to hear the generated melody without adding a full audio stack.

In [ ]:
waveform = melody_tokens_to_waveform(melodies[0], tempo_bpm=120)
print("Waveform samples:", waveform.shape[0])

plt.figure(figsize=(10, 3))
plt.plot(waveform[:4000])
plt.title("First Melody Waveform Preview")
plt.xlabel("Sample index")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()